# Análisis hidrológico de la Cuenca del Río Suquía

Delimitación, red de drenaje, perfil del cauce principal y subcuencas a partir de un
DEM **Copernicus GLO-30**, usando [pysheds](https://github.com/mdbartos/pysheds).

**Requisitos:** `pip install pysheds rasterio numpy scipy matplotlib cartopy requests`


## 0. Imports, estilo y parámetros


In [ ]:
import os, math
import requests
import numpy as np
from scipy import ndimage
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.colors import LightSource, LogNorm, ListedColormap
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pysheds.grid import Grid
import rasterio
from collections import deque
import warnings
warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
# Tema oscuro unificado para TODOS los graficos (mismo fondo, tipografia y paleta)
DARK, PANEL, TXT, MUT, LINE = "#0d0826", "#1a1530", "#e8e8ec", "#c9c9d4", "#555a6e"
plt.rcParams.update({
    "figure.facecolor": DARK, "axes.facecolor": DARK, "savefig.facecolor": DARK,
    "savefig.dpi": 220, "savefig.bbox": "tight",
    "text.color": TXT, "axes.labelcolor": TXT, "axes.titlecolor": "#ffffff",
    "axes.edgecolor": LINE, "axes.linewidth": 0.8,
    "xtick.color": MUT, "ytick.color": MUT, "grid.color": "#3a3550",
    "font.size": 11, "axes.titlesize": 14, "axes.titleweight": "bold", "axes.labelsize": 11,
    "legend.facecolor": PANEL, "legend.edgecolor": LINE, "legend.framealpha": 0.9,
    "legend.fontsize": 9, "legend.labelcolor": TXT,
})

COLORS = {
    "agua":        "#1f6feb",   # lago y cauce principal (agua = azul)
    "naciente":    "#e4572e",   # punto de naciente
    "perfil_fill": "#b08433",   # relleno bajo el perfil (tierra)
    "cuenca":      "#6b6486",   # silueta de cuenca en el inset
    "san_antonio": "#e4572e",
    "cosquin":     "#2e86ab",
    "otros":       "#8ac926",
    "baja":        "#9b5de5",
}


In [ ]:
FN  = "cordoba_cop30_dem.tif"
BBOX = (-65.0, -31.68, -63.6, -31.05)        # ventana de analisis (cuenca del Suquia)
DIRMAP = (64, 128, 1, 2, 4, 8, 16, 32)
OUTLET    = (-63.98, -31.38)                 # salida de la cuenca
SEED_LAGO = (-64.45, -31.37)                 # semilla sobre el embalse San Roque
AREA_PIXEL_KM2 = 0.0009                      # nominal COP30 (~30 m)
Q_ESPECIFICO   = 0.0062                      # m3/s por km2, calibrado al aforo de San Roque

OFFSETS = {64:(-1,0), 128:(-1,1), 1:(0,1), 2:(1,1), 4:(1,0), 8:(1,-1), 16:(0,-1), 32:(-1,-1)}


## 1. Descarga del DEM (Copernicus GLO-30)

Recorte vía la API de [OpenTopography](https://portal.opentopography.org/). Se descarga
la provincia completa; el análisis luego acota a la ventana de la cuenca del Suquía.


In [ ]:
south, north, west, east = -35.5, -29.0, -66.5, -61.5   # provincia de Cordoba
api_key = "TU_API_KEY"   # https://portal.opentopography.org/  ->  NO subas tu clave real al repo

if os.path.exists(FN):
    print(f"'{FN}' ya existe, se omite la descarga.")
else:
    print("Solicitando recorte del DEM Copernicus 30m (puede tardar un par de minutos)...")
    url = (f"https://portal.opentopography.org/API/globaldem?demtype=COP30"
           f"&south={south}&north={north}&west={west}&east={east}"
           f"&format=GTiff&API_Key={api_key}")
    r = requests.get(url)
    if r.status_code == 200:
        with open(FN, "wb") as f:
            f.write(r.content)
        print(f"Listo: '{FN}' guardado.")
    else:
        print(f"Error {r.status_code}: {r.text}")


## 2. Carga y acondicionamiento del DEM

Suaviza, rellena pits y depresiones, y resuelve los flats con `max_iter=1e9, eps=1e-12`
(clave: las superficies planas grandes como el embalse no se resuelven con los valores
por defecto). Calcula direcciones de flujo, acumulación, delimita la cuenca y detecta el
lago. Tarda ~1 minuto.


In [ ]:
with rasterio.open(FN) as src:
    nodata = src.nodata if src.nodata is not None else -9999

grid = Grid.from_raster(FN, nodata=nodata, window=BBOX)
dem  = grid.read_raster(FN, nodata=nodata, window=BBOX)
dem_raw = np.asarray(dem).astype(float)
aff = grid.affine
print("DEM:", dem.shape)

sm = ndimage.gaussian_filter(dem, sigma=0.8)
conditioned = dem.copy(); conditioned[:] = sm
for i in range(5):
    conditioned = grid.fill_pits(conditioned)
    conditioned = grid.fill_depressions(conditioned)
    conditioned = grid.resolve_flats(conditioned, max_iter=int(1e9), eps=1e-12)
    if not grid.detect_pits(conditioned).any() and not grid.detect_flats(conditioned).any():
        break
print("Acondicionado en", i + 1, "pasadas")

flowdir = grid.flowdir(conditioned, dirmap=DIRMAP)
fd  = np.asarray(flowdir)
acc = grid.accumulation(flowdir, dirmap=DIRMAP)
acc_arr = np.asarray(acc)

x_snap, y_snap = grid.snap_to_mask(acc > int(1800 / AREA_PIXEL_KM2), OUTLET)
catch = grid.catchment(x=x_snap, y=y_snap, fdir=flowdir, dirmap=DIRMAP, xytype="coordinate")
catch_arr = np.asarray(catch)

# Embalse San Roque: region plana conexa, sembrada sobre el lago
col0, row0 = grid.nearest_cell(*SEED_LAGO)
rango = ndimage.maximum_filter(dem_raw, size=3) - ndimage.minimum_filter(dem_raw, size=3)
labels, n = ndimage.label(rango < 1.0)
lab = int(labels[row0, col0])
if lab == 0:
    win = labels[max(0, row0-4):row0+5, max(0, col0-4):col0+5]; v = win[win > 0]
    lab = int(np.bincount(v).argmax()) if v.size else 0
lago = (labels == lab) if lab > 0 else np.zeros_like(catch_arr, bool)

rows = np.arange(dem_raw.shape[0])
lat_rows = aff.f + aff.e * (rows + 0.5)
area_fila = (abs(aff.a) * 111320 * np.cos(np.radians(lat_rows))) * (abs(aff.e) * 110540)
print("Cuenca delimitada. Lago:", int(lago.sum()), "celdas")


### Funciones auxiliares (estilo, métricas, perfil, dibujo)


In [ ]:
ls = LightSource(azdeg=315, altdeg=45)

_DR = np.zeros(256, int); _DC = np.zeros(256, int); _OK = np.zeros(256, bool)
for _k, (_a, _b) in OFFSETS.items():
    _DR[_k] = _a; _DC[_k] = _b; _OK[_k] = True

def lonlat(row, col):
    return aff.c + aff.a * (col + 0.5), aff.f + aff.e * (row + 0.5)

def haversine(lo1, la1, lo2, la2):
    R = 6371000.0; p1, p2 = math.radians(la1), math.radians(la2)
    dphi = math.radians(la2 - la1); dl = math.radians(lo2 - lo1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def area_km2(mask):
    return float((np.asarray(mask).sum(axis=1) * area_fila).sum() / 1e6)

def relieve(mask):
    v = dem_raw[np.asarray(mask)]; v = v[(np.isfinite(v)) & (v > -1000)]
    return float(v.max()), float(v.min())

def caudal_m3s(mask):
    return Q_ESPECIFICO * area_km2(mask)

def red(mask=None, umbral_km2=2.5):
    m = catch_arr if mask is None else np.asarray(mask)
    return (acc_arr > int(umbral_km2 / AREA_PIXEL_KM2)) & m & ~lago

def longitud_red_km(streams):
    rr, cc = np.where(streams); fv = np.clip(fd[rr, cc], 0, 255); ok = _OK[fv]
    rr, cc, fv = rr[ok], cc[ok], fv[ok]
    lo1 = aff.c + aff.a*(cc+0.5);            la1 = aff.f + aff.e*(rr+0.5)
    lo2 = aff.c + aff.a*(cc+_DC[fv]+0.5);    la2 = aff.f + aff.e*(rr+_DR[fv]+0.5)
    R = 6371000.0; p1, p2 = np.radians(la1), np.radians(la2)
    a = np.sin(np.radians(la2-la1)/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(np.radians(lo2-lo1)/2)**2
    return float(2*R*np.arcsin(np.sqrt(np.clip(a, 0, 1))).sum()) / 1000.0

def camino_principal(umbral_km2=2.5):
    # BFS inverso. Incluimos el lago en el grafo para que el camino lo ATRAVIESE
    # (si no, la red queda partida en el embalse y no se alcanzan las nacientes altas).
    graph = (red(umbral_km2=umbral_km2) | lago) & catch_arr
    cells = set(map(tuple, np.argwhere(graph)))
    inv = {c: [] for c in cells}
    for (r, c) in cells:
        off = OFFSETS.get(int(fd[r, c]))
        if off and (r+off[0], c+off[1]) in inv:
            inv[(r+off[0], c+off[1])].append((r, c))
    ocol, orow = grid.nearest_cell(x_snap, y_snap); outlet = (orow, ocol)
    dist = {outlet: 0.0}; q = deque([outlet])
    while q:
        r, c = q.popleft(); lo_d, la_d = lonlat(r, c)
        for nr, nc in inv.get((r, c), []):
            if (nr, nc) not in dist:
                lo_u, la_u = lonlat(nr, nc)
                dist[(nr, nc)] = dist[(r, c)] + haversine(lo_u, la_u, lo_d, la_d); q.append((nr, nc))
    r, c = max(dist, key=dist.get); camino = [(r, c)]; visto = {(r, c)}; H, W = fd.shape
    for _ in range(H * W):
        if (r, c) == outlet: break
        off = OFFSETS.get(int(fd[r, c]))
        if not off: break
        r, c = r + off[0], c + off[1]
        if not (0 <= r < H and 0 <= c < W) or (r, c) in visto: break
        visto.add((r, c)); camino.append((r, c))
    d = [0.0]; z = [dem_raw[camino[0]]]
    for i in range(1, len(camino)):
        lo1, la1 = lonlat(*camino[i-1]); lo2, la2 = lonlat(*camino[i])
        d.append(d[-1] + haversine(lo1, la1, lo2, la2)); z.append(dem_raw[camino[i]])
    z = np.array(z); z[z < -1000] = np.nan
    return np.array(d) / 1000.0, z, camino

def encuadre(pad=0.08):
    ys, xs = np.where(catch_arr)
    ll = aff.c + aff.a*xs.min(); lr = aff.c + aff.a*(xs.max()+1)
    lt = aff.f + aff.e*ys.min(); lb = aff.f + aff.e*(ys.max()+1)
    return [ll-pad, lr+pad, lb-pad, lt+pad]

def figura(ext):
    asp = (ext[1]-ext[0]) / (ext[3]-ext[2]); obj = 12.0
    w, h = (obj, obj/asp) if asp >= 1 else (obj*asp, obj)
    return plt.subplots(figsize=(w, h), subplot_kw={"projection": ccrs.PlateCarree()})

def grilla(ax):
    gl = ax.gridlines(draw_labels=["bottom", "left"], dms=True, alpha=0.2, color="white")
    gl.xlabel_style = {"color": MUT, "size": 9}
    gl.ylabel_style = {"color": MUT, "size": 9}

def vista_raster(data, cmap, titulo, label=None, ext=None, guardar=None):
    if ext is None:
        fig, ax = plt.subplots(figsize=(13, 5.5), subplot_kw={"projection": ccrs.PlateCarree()})
    else:
        fig, ax = figura(ext)
    im = ax.imshow(np.asarray(data), extent=grid.extent, cmap=cmap,
                   interpolation="bilinear", transform=ccrs.PlateCarree())
    if label:
        cb = plt.colorbar(im, ax=ax, label=label, shrink=0.8, pad=0.02)
    grilla(ax); ax.set_title(titulo, pad=12)
    if ext is not None:
        ax.set_extent(ext, crs=ccrs.PlateCarree())
    plt.tight_layout()
    if guardar:
        plt.savefig(guardar)
    plt.show()

CIUDADES = {
    "Cordoba Cap.": (-64.188, -31.413, 0.02, -0.05),
    "La Calera":    (-64.335, -31.343, 0.02,  0.03),
    "Carlos Paz":   (-64.494, -31.424, -0.10, -0.05),
    "Cosquin":      (-64.466, -31.245, -0.12, -0.01),
    "La Falda":     (-64.489, -31.092, -0.11,  0.01),
}
def dibujar_ciudades(ax, ext, cuales=None):
    for nombre, (lon, lat, dx, dy) in CIUDADES.items():
        if cuales and nombre not in cuales: continue
        if ext[0] <= lon <= ext[1] and ext[2] <= lat <= ext[3]:
            ax.scatter(lon, lat, c="white", edgecolors="black", s=45, zorder=6, transform=ccrs.PlateCarree())
            ax.text(lon+dx, lat+dy, nombre, fontsize=10, fontweight="bold", color="white",
                    zorder=7, transform=ccrs.PlateCarree())


## 3. Vistas del DEM, relieve y direcciones de flujo


In [ ]:
vista_raster(np.where(dem_raw > -1000, dem_raw, np.nan), "terrain",
             "Modelo de Elevación Digital - Copernicus GLO-30 (30 m)", "Elevación (m s.n.m.)", ext=encuadre(), guardar="dem_original.png")


In [ ]:
vista_raster(ls.hillshade(conditioned, fraction=1.0), "gray",
             "Relieve sombreado (hillshade)", ext=encuadre(), guardar="hillshade.png")


In [ ]:
vista_raster(np.where(catch_arr, fd.astype(float), np.nan), "twilight",
             "Direcciones de flujo (D8)", "Código de dirección", ext=encuadre(), guardar="flujo_direccion.png")


## 4. Mapa de la red de drenaje

Criterio: celdas con área de acumulación > 2.5 km², dentro de la cuenca, sin el espejo del lago.


In [ ]:
umbral_red_km2 = 2.5; grosor = 2
streams = red(umbral_km2=umbral_red_km2)
red_acc = np.where(ndimage.binary_dilation(streams, iterations=grosor), acc_arr, np.nan)
shaded  = np.where(catch_arr, ls.hillshade(conditioned, fraction=1.0), np.nan)

ext = encuadre(); fig, ax = figura(ext)
ax.imshow(shaded, cmap="gray", extent=grid.extent, alpha=0.35, interpolation="bilinear",
          transform=ccrs.PlateCarree(), zorder=1)
im = ax.imshow(red_acc, extent=grid.extent, cmap="autumn",
               norm=LogNorm(int(umbral_red_km2/AREA_PIXEL_KM2), acc_arr.max()),
               interpolation="none", transform=ccrs.PlateCarree(), zorder=2)
ax.imshow(np.where(lago, 1, np.nan), extent=grid.extent, cmap=ListedColormap([COLORS["agua"]]),
          alpha=0.95, interpolation="none", transform=ccrs.PlateCarree(), zorder=3)
plt.colorbar(im, ax=ax, label="Área de acumulación (celdas aguas arriba)", shrink=0.85, pad=0.02)
dibujar_ciudades(ax, ext)
ax.add_feature(cfeature.BORDERS, linestyle="--", alpha=0.5)
grilla(ax)
ax.set_title("Cuenca del Río Suquía - Red de drenaje", pad=14)
ax.set_extent(ext, crs=ccrs.PlateCarree())
plt.tight_layout(); plt.savefig("mapa_red.png"); plt.show()


## 5. Perfil del cauce principal y métricas globales

El cauce principal es el camino de flujo más largo hasta la salida (atraviesa el embalse, que aparece como tramo plano). El panel izquierdo muestra su trazado.


In [ ]:
dkm, zm, camino = camino_principal()
L = float(dkm[-1]); z0 = float(np.nanmax(zm[:5])); zf = float(np.nanmin(zm[-5:]))
dz = z0 - zf; pend = dz / (L * 1000) * 100

streams = red(umbral_km2=2.5); km_red = longitud_red_km(streams)
area = area_km2(catch_arr); zmax, zmin = relieve(catch_arr); Q = caudal_m3s(catch_arr)

print(f"Area total            : {area:,.0f} km2")
print(f"Longitud de red        : {km_red:,.0f} km")
print(f"Cota max / min         : {zmax:.0f} / {zmin:.0f} m  (relieve {zmax-zmin:.0f} m)")
print(f"Cauce principal        : {L:.1f} km | salto {dz:.0f} m | pendiente {pend:.2f} %")
print(f"Caudal medio estimado  : ~{Q:.1f} m3/s")

fig = plt.figure(figsize=(14, 5))
gs = fig.add_gridspec(1, 4, wspace=0.28)
axm = fig.add_subplot(gs[0, 0]); axp = fig.add_subplot(gs[0, 1:])

# Panel izquierdo: trazado del cauce principal sobre la cuenca
ys, xs = np.where(catch_arr); r0, r1, c0, c1 = ys.min(), ys.max(), xs.min(), xs.max()
axm.imshow(np.where(catch_arr[r0:r1+1, c0:c1+1], 1, np.nan),
           cmap=ListedColormap([COLORS["cuenca"]]), origin="upper", interpolation="none")
axm.plot([p[1]-c0 for p in camino], [p[0]-r0 for p in camino], color=COLORS["agua"], lw=1.8)
axm.scatter(camino[0][1]-c0, camino[0][0]-r0, color=COLORS["naciente"], s=28, zorder=3)
axm.set_xticks([]); axm.set_yticks([]); axm.set_title("Cauce principal tomado", fontsize=10)
for sp in axm.spines.values(): sp.set_edgecolor(LINE)

# Panel derecho: perfil
base = float(np.nanmin(zm)) - 40
axp.fill_between(dkm, zm, base, color=COLORS["perfil_fill"], alpha=0.30)
axp.plot(dkm, zm, color=COLORS["agua"], lw=2.2)
axp.set_xlabel("Distancia desde la naciente más lejana (km)")
axp.set_ylabel("Elevación (m s.n.m.)")
axp.set_title("Perfil longitudinal del cauce principal - Río Suquía")
axp.grid(alpha=0.25); axp.margins(x=0); axp.set_ylim(base, None)
stats = (f"Longitud : {L:.1f} km\nSalto    : {dz:.0f} m\nPendiente: {pend:.2f} %\n"
         f"Área     : {area:,.0f} km²\nCaudal   : ~{Q:.1f} m³/s")
axp.text(0.985, 0.95, stats, transform=axp.transAxes, ha="right", va="top", family="monospace",
         color=TXT, bbox=dict(boxstyle="round", fc=PANEL, ec=LINE, alpha=0.9))

plt.tight_layout(); plt.savefig("perfil_suquia.png"); plt.show()


## 6. Subcuencas

San Antonio y Cosquín por punto de cierre. El aporte al San Roque por propagación
topológica descendente desde el lago. Otros aportes y cuenca baja, por diferencia.


In [ ]:
umbral_snap = int(40 / AREA_PIXEL_KM2)
def cuenca_en(x, y):
    xs, ys = grid.snap_to_mask(acc > umbral_snap, (x, y))
    return np.asarray(grid.catchment(x=xs, y=ys, fdir=flowdir, dirmap=DIRMAP, xytype="coordinate")) & catch_arr

san_antonio = cuenca_en(-64.503, -31.435)
cosquin     = cuenca_en(-64.462, -31.330)

H, W = catch_arr.shape
rr, cc = np.where(catch_arr); order = np.argsort(-acc_arr[rr, cc])
rs, cs = rr[order], cc[order]; fv = np.clip(fd[rs, cs], 0, 255); ok = _OK[fv]
dr_ = np.clip(rs + _DR[fv], 0, H-1); dc_ = np.clip(cs + _DC[fv], 0, W-1)
aporte = lago.copy().astype(bool)
for i in range(len(rs)):
    if ok[i] and aporte[dr_[i], dc_[i]]:
        aporte[rs[i], cs[i]] = True

sub = {
    "Río San Antonio":          san_antonio & ~cosquin,
    "Río Cosquín":              cosquin & ~san_antonio,
    "Otros aportes al lago":    aporte & ~(san_antonio | cosquin),
    "Cuenca baja (post dique)": catch_arr & ~aporte,
}
colores = {"Río San Antonio": COLORS["san_antonio"], "Río Cosquín": COLORS["cosquin"],
           "Otros aportes al lago": COLORS["otros"], "Cuenca baja (post dique)": COLORS["baja"]}
orden = list(sub.keys())

print(f"{'Subcuenca':<28}{'Area km2':>10}{'Red km':>9}{'Relieve m':>11}{'Q m3/s':>9}")
print("-" * 67)
for k in orden:
    m = sub[k]; zx, zn = relieve(m)
    print(f"{k:<28}{area_km2(m):>10,.0f}{longitud_red_km(red(mask=m)):>9,.0f}{zx-zn:>11.0f}{caudal_m3s(m):>9.1f}")
print("-" * 67)
print(f"Validación aporte San Roque: {area_km2(aporte):,.0f} km2  (ref. INA ~1750 km2)")


In [ ]:
shade = 0.35 + 0.65 * ls.hillshade(conditioned, fraction=1.0)
rgba = np.zeros((*catch_arr.shape, 4), np.float32)
for k in orden:
    m = np.asarray(sub[k]); r, g, b = mcolors.to_rgb(colores[k]); br = shade * m
    rgba[..., 0] += br*r; rgba[..., 1] += br*g; rgba[..., 2] += br*b; rgba[..., 3] += m
canal = red(umbral_km2=3.0) & ~lago
rgba[canal, :3] = np.clip(rgba[canal, :3] + 0.30, 0, 1)
rl, gl, bl = mcolors.to_rgb(COLORS["agua"])
rgba[lago, 0] = rl; rgba[lago, 1] = gl; rgba[lago, 2] = bl; rgba[lago, 3] = 1.0
for k in orden:
    edge = ndimage.binary_dilation(np.asarray(sub[k])) & ~np.asarray(sub[k]) & catch_arr
    rgba[edge, :3] = 0.12; rgba[edge, 3] = 1.0
rgba[~catch_arr, 3] = 0.0; rgba = np.clip(rgba, 0, 1)

ext = encuadre(); fig, ax = figura(ext)
ax.imshow(rgba, extent=grid.extent, interpolation="bilinear", transform=ccrs.PlateCarree(), zorder=1)
dibujar_ciudades(ax, ext, cuales={"Cordoba Cap.", "Carlos Paz", "Cosquin", "La Falda"})
handles = [mpatches.Patch(color=colores[k], label=f"{k}  ({area_km2(sub[k]):,.0f} km²)") for k in orden]
leg = ax.legend(handles=handles, loc="lower left", title="Subcuencas")
leg.get_title().set_color(TXT)
grilla(ax)
ax.set_title("Subcuencas del Río Suquía (aporte al dique San Roque)", pad=14)
ax.set_extent(ext, crs=ccrs.PlateCarree())
plt.tight_layout(); plt.savefig("mapa_subcuencas.png"); plt.show()


## 7. Sobre el cálculo de caudales

Del DEM no se puede sacar caudal directamente: da área y pendiente, no volumen. La
estimación usa un **caudal específico calibrado a un aforo real**: el módulo del Suquía
en San Roque es ~10,9 m³/s sobre ~1750 km² (INA), o sea **q ≈ 0,0062 m³/s/km²**. El
caudal de cada cuenca se estima como **Q ≈ q · área**.

Es una estimación de orden de magnitud. El `q` está calibrado a la cuenca serrana
(húmeda); aplicarlo a la cuenca baja (árida, regulada por el dique) la sobreestima. Un
caudal confiable requiere series de precipitación/escorrentía y un modelo hidrológico.

**Fuentes:** DEM Copernicus GLO-30 (ESA); caudal y subcuencas de referencia: INA / CIRSA.
